# 05 - Model Evaluation
Compare all models, visualize ROC/PR curves, confusion matrix, and classification report.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             confusion_matrix, classification_report,
                             RocCurveDisplay, PrecisionRecallDisplay)

from config import PROCESSED_DATA_DIR, MODELS_DIR, RANDOM_STATE, TEST_SIZE
from src.util import load_dataframe, load_model

## 5.1 Load data and artifacts

In [ ]:
df = load_dataframe(os.path.join(PROCESSED_DATA_DIR, 'df_engineered.csv'))

# load training artifacts
artifacts = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'training_artifacts.pkl'))
selected = artifacts['selected_features']
results = artifacts['results']
best_name = artifacts['best_model_name']

print(f'Data shape: {df.shape}')
print(f'Selected features: {len(selected)}')
print(f'Best model: {best_name}')

In [ ]:
# re-create train/test split with same random state
X = df[selected].copy().fillna(0).replace([np.inf, -np.inf], 0)
y = df['is_churned'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# scale for logistic regression
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=selected, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=selected, index=X_test.index)

print(f'Test set: {X_test.shape[0]:,} rows')

In [ ]:
# load all saved models
model_files = {
    'Logistic Regression': 'logistic_regression.pkl',
    'Random Forest': 'random_forest.pkl',
    'XGBoost': 'xgboost.pkl',
    'LightGBM': 'lightgbm.pkl',
}

models = {}
for name, fname in model_files.items():
    path = os.path.join(MODELS_DIR, fname)
    if os.path.exists(path):
        models[name] = load_model(path)

print(f'\nLoaded {len(models)} models: {list(models.keys())}')

## 5.2 Model comparison table

In [ ]:
comparison = pd.DataFrame(results).T.round(4)
styled = comparison.style.highlight_max(axis=0, color='lightgreen')
display(styled)

## 5.3 ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for (name, model), color in zip(models.items(), colors):
    if name == 'Logistic Regression':
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.predict_proba(X_test)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - All Models')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5.4 Precision-Recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for (name, model), color in zip(models.items(), colors):
    if name == 'Logistic Regression':
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.predict_proba(X_test)[:, 1]
    
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ax.plot(recall, precision, color=color, lw=2, label=f'{name}')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves - All Models')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## 5.5 Confusion matrix (best model)

In [ ]:
best = models[best_name]
if best_name == 'Logistic Regression':
    y_pred = best.predict(X_test_scaled)
else:
    y_pred = best.predict(X_test)

# raw confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Saved', 'Lost'], yticklabels=['Saved', 'Lost'])
axes[0].set_title(f'Confusion Matrix - {best_name} (Counts)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['Saved', 'Lost'], yticklabels=['Saved', 'Lost'])
axes[1].set_title(f'Confusion Matrix - {best_name} (Normalized)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 5.6 Classification report

In [ ]:
print(f'Classification Report - {best_name}:\n')
print(classification_report(y_test, y_pred, target_names=['Customer Saved', 'Customer Lost']))